# task 2 -> proximal policy optimization

## imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.data import load_yaml, read_jsonl, repo_path
from task2_ppo.continue_train import fork_name

## global env variables ->

In [ ]:
#papermill overrides CONFIG for the smoke test, the defaults are the real run
CONFIG = 'configs/ppo.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir'])
FIGURES = RESULTS / 'figures'
STANDARD = cfg['output']
MIDPOINT = cfg['paths']['ppo_midpoint_policy']

EPS_VALUES = [float(eps) for eps in cfg['clip_values']]
KL_VALUES = [float(beta) for beta in cfg['kl_values']]
#the forks of each study. the eps 0.2 / beta 0.1 fork belongs to both, it is trained once and read twice
EPS_FORKS = {eps: fork_name(eps, float(cfg['kl_beta'])) for eps in EPS_VALUES}
KL_FORKS = {beta: fork_name(float(cfg['clip_epsilon']), beta) for beta in KL_VALUES}

#one fixed colour per condition inside each study, the reference/midpoint rows stay neutral grey
EPS_COLORS = dict(zip(EPS_VALUES, ['#2a78d6', '#eb6834', '#1baf7a']))
KL_COLORS = dict(zip(KL_VALUES, ['#eda100', '#e87ba4', '#6250d6']))
INK, MUTED = '#0b0b0b', '#52514e'

PYTHON = sys.executable #the interpreter running this notebook, so the scripts use the same environment

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'lines.linewidth': 2})


def run(command):
    #runs one of the release's entry points as its own process, exactly as it would be typed in a terminal.
    #each script loads its own models and writes its results to disk, nothing is passed through notebook state
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'


def load_result(name):
    return json.loads((RESULTS / name).read_text())


def train_log(name):
    return pd.DataFrame(read_jsonl(RESULTS / f'train_log_{name}.jsonl'))


def heldout_row(name):
    #the held-out numbers of one policy, the same columns for every table below
    ev = load_result(f'eval_{name}.json')
    return {'heldout_reward': ev['reward_mean'], 'heldout_kl_token': ev['kl_token_mean'], 'heldout_kl_sequence': ev['kl_sequence_mean'],
            'heldout_entropy': ev['entropy_mean'], 'heldout_length': ev['length_mean'], 'heldout_length_std': ev['length_std'],
            'heldout_truncation': ev['truncation_rate']}

## standard ppo continuation ->

20 updates from the supplied midpoint policy and critic with the release configuration

In [ ]:
run(f'{PYTHON} -m task2_ppo.continue_train --config {CONFIG} --run-name standard --output {STANDARD}')

held-out evaluation of the frozen reference, the supplied midpoint and the continued policy, same prompts and decoding

In [ ]:
run(f'{PYTHON} -m task2_ppo.evaluate --config {CONFIG} --name reference')
run(f'{PYTHON} -m task2_ppo.evaluate --config {CONFIG} --adapter {MIDPOINT} --name midpoint')
run(f'{PYTHON} -m task2_ppo.evaluate --config {CONFIG} --adapter {STANDARD} --name standard')

## clipping study ->

the cached rollout batch at eps 0.05 / 0.2 / 0.5, then one short fork per eps from the same midpoint

In [ ]:
run(f'{PYTHON} -m task2_ppo.analyze_clipping --config {CONFIG}')

## reward overoptimization study ->

one short fork per kl beta from the same midpoint, only the kl penalty changes

In [ ]:
run(f'{PYTHON} -m task2_ppo.ablate_kl --config {CONFIG}')

## standard continuation diagnostics

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
standard_log = train_log('standard')
standard_summary = load_result('train_summary_standard.json')

#the compute the manual asks to be reported for the standard continuation
display(pd.DataFrame([{
    'gpu': standard_summary['gpu'],
    'updates': standard_summary['updates'],
    'generated_tokens': standard_summary['generated_tokens'],
    'wall_clock_min': standard_summary['wall_clock_s'] / 60,
    'peak_vram_gb': standard_summary['peak_vram_gb'],
}]).round(2))

columns = ['update', 'reward', 'reward_effective', 'kl', 'policy_loss', 'value_loss', 'entropy', 'clip_fraction',
           'grad_norm', 'response_length', 'truncated', 'step_kl']
standard_log[columns].to_csv(RESULTS / 'standard_trajectory.csv', index=False)
display(standard_log[columns].round(4))

In [ ]:
#one panel per logged quantity, each on its own scale. every update is a single prompt (prompts_per_update = 1), so
#the thin line is the raw per-update value and the thick line a 5 update rolling mean to show the trend
panels = [('reward', 'learned reward'), ('kl', 'kl from reference (per token)'), ('policy_loss', 'policy loss'),
          ('value_loss', 'value loss (before fitting)'), ('entropy', 'policy entropy'), ('clip_fraction', 'clip fraction'),
          ('grad_norm', 'policy gradient norm'), ('response_length', 'response length (tokens)')]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for axis, (column, title) in zip(axes.ravel(), panels):
    axis.plot(standard_log['update'], standard_log[column], color='#9ec5f4', linewidth=1.2)
    axis.plot(standard_log['update'], standard_log[column].rolling(5, min_periods=1).mean(), color='#2a78d6')
    axis.set_title(title)
    axis.set_xlabel('update')
    axis.set_xticks(range(0, len(standard_log) + 1, 5))
axes[1, 3].axhline(int(cfg['max_response_length']), color=MUTED, linestyle=':', linewidth=1) #the 512 token rollout cap
fig.tight_layout()
fig.savefig(FIGURES / 'standard_trajectories.png', dpi=200)
plt.show()

In [ ]:
#critic behaviour -> what it predicted for each rollout against what that rollout actually earned.
#value_mean is the critic's prediction averaged over the response tokens. the realised outcome is reward_effective, the
#terminal reward the episode really received (gamma = 1, so every token's true return is that reward minus the small
#kl penalties still to come). return_mean is NOT used as the outcome -> the gae return is advantage + the critic's own
#value, so it follows the critic by construction and would make any critic look accurate
critic = standard_log[['update', 'value_mean', 'reward_effective', 'return_mean', 'value_loss', 'value_loss_final',
                       'explained_variance', 'value_grad_norm']]
critic.to_csv(RESULTS / 'standard_critic.csv', index=False)
display(critic.round(3))
print('correlation of the critic prediction with the realised reward across updates',
      round(standard_log['value_mean'].corr(standard_log['reward_effective']), 3))
print('mean absolute error of the prediction', round((standard_log['value_mean'] - standard_log['reward_effective']).abs().mean(), 3),
      '  against a reward spread (std) of', round(standard_log['reward_effective'].std(), 3))
print('correlation with the gae return, which contains the critic itself',
      round(standard_log['value_mean'].corr(standard_log['return_mean']), 3))

fig, axis = plt.subplots(figsize=(8, 4.5))
axis.plot(standard_log['update'], standard_log['reward_effective'], color='#2a78d6', marker='o', markersize=6, label='realised reward')
axis.plot(standard_log['update'], standard_log['value_mean'], color='#eb6834', marker='o', markersize=6, label='critic prediction')
axis.set_xlabel('update')
axis.set_title('critic prediction against the realised reward, per rollout')
axis.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / 'standard_critic.png', dpi=200)
plt.show()

In [ ]:
#held-out behaviour before and after the continuation, with the frozen reference as the anchor
heldout = pd.DataFrame([{'policy': name, **heldout_row(name)} for name in ['reference', 'midpoint', 'standard']])
heldout.to_csv(RESULTS / 'standard_heldout.csv', index=False)
display(heldout.round(4))

## clipping study results

In [ ]:
#cached rollout batch -> the immediate geometric effect of eps on one fixed batch.
#pi_old is the supplied midpoint re-scored on the cached tokens, pi_new the midpoint after one unclipped pass of
#policy gradient steps over the batch, so the ratio is a real policy change measured with consistent numerics
display(pd.read_csv(RESULTS / 'cached_clip_study.csv').round(5))

#why the cached old log probs are not used as pi_old directly -> 'direct' takes them at face value, 'control' repeats
#the same computation with the frozen reference, whose true ratio is exactly 1. the control clips about as many
#tokens as the direct table when the difference to the cache is numerical noise rather than a policy change
display(pd.Series(load_result('cached_clip_consistency.json')).to_frame('value'))
noise = pd.concat([pd.read_csv(RESULTS / 'cached_clip_direct.csv').assign(table='direct (cached old -> midpoint)'),
                   pd.read_csv(RESULTS / 'cached_clip_reference_control.csv').assign(table='control (cached ref -> reference)')])
display(noise[['table', 'eps', 'clip_fraction', 'affected_token_fraction', 'ratio_min', 'ratio_max']].round(5))

In [ ]:
#matched short forks -> held-out behaviour plus the stability statistics of each fork's own updates.
#step_kl = sampled kl(pi_old || pi_new) on the update's rollout after its ppo epochs, ie how far one update moved the
#policy. its mean and max over the fork, and the largest single token |log ratio|, are the stability statistics
def fork_table(forks, key):
    rows = []
    for value, name in forks.items():
        log, summary = train_log(name), load_result(f'train_summary_{name}.json')
        rows.append({
            key: value, 'fork': name, **heldout_row(name),
            'step_kl_mean': log['step_kl'].mean(), 'step_kl_max': log['step_kl'].max(),
            'max_abs_log_ratio': log['max_abs_log_ratio'].max(),
            'clip_fraction_mean': log['clip_fraction'].mean(),
            'grad_norm_mean': log['grad_norm'].mean(), 'grad_norm_std': log['grad_norm'].std(ddof=0),
            'train_reward_mean': log['reward'].mean(), 'train_kl_mean': log['kl'].mean(),
            'train_entropy_mean': log['entropy'].mean(), 'train_length_mean': log['response_length'].mean(),
            'updates': summary['updates'], 'generated_tokens': summary['generated_tokens'],
        })
    return pd.DataFrame(rows)


midpoint_row = heldout_row('midpoint')
eps_table = fork_table(EPS_FORKS, 'eps')
eps_table.to_csv(RESULTS / 'clipping_forks.csv', index=False)
display(eps_table.round(5))

In [ ]:
def fork_trajectories(forks, colors, label, columns, filename):
    #the logged per-update trajectories of each fork. all forks see the same prompt at the same update, so the
    #lines are directly comparable update by update
    fig, axes = plt.subplots(1, len(columns), figsize=(5 * len(columns), 4))
    for axis, (column, title) in zip(axes, columns):
        for value, name in forks.items():
            log = train_log(name)
            axis.plot(log['update'], log[column], color=colors[value], marker='o', markersize=5, label=f'{label} {value:g}')
        axis.set_title(title)
        axis.set_xlabel('update')
    axes[0].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=200)
    plt.show()


fork_trajectories(EPS_FORKS, EPS_COLORS, 'eps',
                  [('clip_fraction', 'clip fraction'), ('step_kl', 'policy change per update (step kl)'),
                   ('grad_norm', 'policy gradient norm'), ('reward', 'learned reward')], 'clipping_fork_trajectories.png')

## kl pressure results

In [ ]:
kl_table = fork_table(KL_FORKS, 'kl_beta')
kl_table.to_csv(RESULTS / 'kl_forks.csv', index=False)
display(pd.DataFrame([{'kl_beta': 'midpoint (start)', **midpoint_row}]).round(5))
display(kl_table.round(5))

fork_trajectories(KL_FORKS, KL_COLORS, 'beta',
                  [('reward', 'learned reward'), ('kl', 'kl from reference (per token)'),
                   ('entropy', 'policy entropy'), ('response_length', 'response length (tokens)')], 'kl_fork_trajectories.png')

In [ ]:
#held-out reward against held-out kl for every policy of the task -> is extra reward bought with extra drift
points = [('reference', 'reference', MUTED), ('midpoint', 'midpoint', INK), ('standard', 'standard (20 updates)', '#008300')]
points += [(name, f'eps {eps:g}', EPS_COLORS[eps]) for eps, name in EPS_FORKS.items()]
points += [(name, f'beta {beta:g}', KL_COLORS[beta]) for beta, name in KL_FORKS.items() if name not in EPS_FORKS.values()]

fig, axis = plt.subplots(figsize=(8, 5.5))
for name, label, color in points:
    row = heldout_row(name)
    axis.scatter(row['heldout_kl_token'], row['heldout_reward'], s=90, color=color, edgecolor='#fcfcfb', linewidth=2, label=label, zorder=3)
axis.set_xlabel('held-out kl from reference (per token)')
axis.set_ylabel('held-out reward model score')
axis.set_title('reward against drift on the held-out prompts')
axis.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
fig.savefig(FIGURES / 'reward_vs_kl.png', dpi=200)
plt.show()

## qualitative candidates ->

shortlists to read through and pick the report examples from, nothing here is selected automatically

In [ ]:
pd.set_option('display.max_colwidth', 500)

#the same held-out prompt answered by the midpoint and by the fork with no kl penalty (the one most free to chase
#reward), sorted by reward gain. large gains -> read whether the response really got better (reward and quality move
#together) or only longer / more padded / truncated (they disagree)
free_fork = KL_FORKS[min(KL_VALUES)]
before = pd.DataFrame(read_jsonl(RESULTS / 'eval_midpoint_generations.jsonl'))
after = pd.DataFrame(read_jsonl(RESULTS / f'eval_{free_fork}_generations.jsonl'))
paired = before.merge(after, on='prompt_id', suffixes=('_midpoint', '_fork'))
paired['reward_gain'] = paired['reward_fork'] - paired['reward_midpoint']
paired['length_change'] = paired['response_tokens_fork'] - paired['response_tokens_midpoint']
columns = ['prompt_id', 'reward_midpoint', 'reward_fork', 'reward_gain', 'response_tokens_midpoint', 'response_tokens_fork',
           'truncated_fork', 'prompt_midpoint', 'response_midpoint', 'response_fork']
print(f'midpoint against {free_fork}, largest reward gains')
display(paired.sort_values('reward_gain', ascending=False)[columns].head(8))
print('largest reward losses')
display(paired.sort_values('reward_gain')[columns].head(5))
print('correlation of reward gain with length change', round(paired['reward_gain'].corr(paired['length_change']), 3))

In [ ]:
#reward against length and termination across all held-out responses of one policy -> does the reward model favour
#long or unfinished answers regardless of content
for name in ['reference', 'midpoint', 'standard', free_fork]:
    frame = pd.DataFrame(read_jsonl(RESULTS / f'eval_{name}_generations.jsonl'))
    print(f"{name:22s} corr(reward, length) {frame['reward'].corr(frame['response_tokens']):+.3f}   "
          f"mean reward finished {frame[~frame['truncated']]['reward'].mean():+.3f}   "
          f"truncated {frame[frame['truncated']]['reward'].mean():+.3f}   ({int(frame['truncated'].sum())} truncated)")